In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchmetrics
import torchcodec
import torchaudio
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, random_split

In [58]:
no_path = Path('datasets/yes_no/250_no.wav')
yes_path = Path('datasets/yes_no/250_yes.wav')

no_wav = torchaudio.load(no_path)
yes_wav = torchaudio.load(yes_path)

In [59]:
no_wav_mono = no_wav[0].mean(dim=0)
yes_wav_mono = yes_wav[0].mean(dim=0)

In [60]:
vad = torchaudio.transforms.Vad(no_wav[1])
chopped_no = vad(no_wav_mono)

vad = torchaudio.transforms.Vad(yes_wav[1])
chopped_yes = vad(yes_wav_mono)

In [61]:
mel = torchaudio.transforms.MelSpectrogram(no_wav[1])
no_spectr = mel(chopped_no)

mel = torchaudio.transforms.MelSpectrogram(yes_wav[1])
yes_spectr = mel(chopped_yes)

/home/damian/New Folder/.venv/lib/python3.14/site-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [62]:
no_spectr_processed = no_spectr[(no_spectr!=0).any(dim=1)]
no_spectr_processed = no_spectr_processed[:, :37911]
no_spectr_processed = torch.cat([no_spectr_processed, torch.zeros(37911).reshape(1, 37911)], dim=0)
print(no_spectr_processed.shape)

yes_spectr_processed = yes_spectr[(yes_spectr!=0).any(dim=1)]
yes_spectr_processed = torch.cat([yes_spectr_processed, torch.ones(37911).reshape(1, 37911)], dim=0)
print(yes_spectr_processed.shape)

torch.Size([113, 37911])
torch.Size([113, 37911])


In [63]:
class TimeSeries(torch.utils.data.Dataset):
    def __init__(self, series, window_length):
        self.series = series
        self.window_length = window_length
    
    def __len__(self):
        return len(self.series[0]) - self.window_length
    
    def __getitem__(self, index):
        if index > len(self):
            raise IndexError('dataset index is out of range')
        end = index + self.window_length
        window = self.series[:112, index:end]
        if self.series[112, 0] == 1.0:
            target = torch.ones((1, 128))
        else:
            target = torch.zeros((1, 128))
        #target = self.series[112, 0]
        return window, target

In [64]:
no_series  = TimeSeries(no_spectr_processed, 128)
yes_series = TimeSeries(yes_spectr_processed, 128)

class TimeSeriesCombined(TimeSeries):
    def __init__(self, series, window_length):
        self.series_0 = series[0]
        self.series_1 = series[1]
        self.window_length = window_length
        self.items = []
        for i in range(self.series_0.__len__()):
            self.items += [self.series_0.__getitem__(i), self.series_1.__getitem__(i)]
            
    def __len__(self):
        return self.series_0.__len__() * 2
    
    def __getitem__(self, index):
        if index > len(self):
            raise IndexError('dataset index is out of range')
        return self.items[index][0], self.items[index][1]

In [65]:
yes_no_dataset = TimeSeriesCombined((no_series, yes_series), 128)

In [66]:
yes_no_train_dataset, yes_no_test_dataset = random_split(yes_no_dataset, [0.9, 0.1])
yes_no_train_dataset, yes_no_valid_dataset = random_split(yes_no_train_dataset, [0.85, 0.15])

train_loader = DataLoader(yes_no_train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(yes_no_valid_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(yes_no_test_dataset, batch_size=16, shuffle=True)

In [67]:
for X, y in train_loader:
#   print(X)
    print(X[0].shape)
    break

torch.Size([112, 128])


In [68]:
class CasualConv1d(nn.Conv1d):
    def forward(self, X):
        padding = (self.kernel_size[0] - 1) * self.dilation[0]
        X = F.pad(X, (padding, 0))
        return super().forward(X)

In [76]:
class rnn(nn.Module):
    def __init__(self, n_inputs=112, conv_dim=32, lstm_dim=64):
        super().__init__()
        conv_layers = []
        for n in range(4):
            conv_layers += [CasualConv1d(n_inputs, conv_dim, kernel_size=2, dilation=2**n), nn.ReLU(), nn.BatchNorm1d(conv_dim)]
            n_inputs = conv_dim
        
        self.conv_stack = nn.Sequential(*conv_layers)
        self.lstm = nn.LSTM(input_size=conv_dim, hidden_size=lstm_dim, batch_first=True)
        self.output = nn.Linear(in_features=lstm_dim, out_features=1)
        
    def forward(self, X):
        #print(X.shape, "#1")
        #Z = X.transpose(1, 2)
        #print(Z.shape, "#2")
        Z = self.conv_stack(X)
        #print(Z.shape, "#3")
        Z = Z.transpose(1, 2)
        #print(Z.shape, '#4')
        Z, _ = self.lstm(Z)
        #print(Z.shape, "#5")
        Z = self.output(Z)
        #print(Z.shape, "#6")
        return Z.transpose(1, 2)

In [1]:
def evaluate(model, metric, data_loader):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
            
        return metric.compute()
    
def train(model, optimizer, criterion, data_loader, valid_loader, metric, n_epochs=50, patience=10, factor=0.1):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='min', patience=patience, factor=factor)
    history = {'train_losses':[], 'train_metrics':[], 'valid_metrics':[]}
    for epoch in range(n_epochs):
        metric.reset()
        model.train()
        total_loss = 0
        for X_batch, y_batch in data_loader:
            if X_batch.shape[2] != 128 or X_batch.shape[0] != 16:
                continue
            X_batch, y_batch = X_batch.to('cuda'), y_batch.to('cuda')
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        
        history['train_losses'].append(total_loss / len(data_loader))
        history['train_metrics'].append(metric.compute().item())
        val_metric = evaluate(model, metric, valid_loader).item()
        history['valid_metrics'].append(val_metric)
        scheduler.step(val_metric)
        
        print(f'Epoch: {epoch+1}\tLoss: {history['train_losses'][-1]:.3f}\tMetrics: {history['train_metrics'][-1]:.3f}\tValid: {history['valid_metrics'][-1]:.3f}')
        
    return history

In [78]:
model_1 = rnn().to('cuda')
metric = torchmetrics.Accuracy(task='binary').to('cuda')
optimizer = torch.optim.NAdam(params=model_1.parameters(), lr=0.001, momentum_decay=0.9)
bxentropy = torch.nn.BCEWithLogitsLoss()

In [80]:
train(model_1, optimizer, bxentropy, train_loader, valid_loader, metric, n_epochs=5)

Epoch: 1	Loss: 0.007	Metrics: 0.997	Valid: 0.997
Epoch: 2	Loss: 0.007	Metrics: 0.997	Valid: 0.997
Epoch: 3	Loss: 0.007	Metrics: 0.997	Valid: 0.996
Epoch: 4	Loss: 0.006	Metrics: 0.997	Valid: 0.997
Epoch: 5	Loss: 0.006	Metrics: 0.997	Valid: 0.998


{'train_losses': [0.007090201895718085,
  0.006872181024693708,
  0.00691668089381057,
  0.006455417399820991,
  0.00648114637801799],
 'train_metrics': [0.9971598982810974,
  0.9972705841064453,
  0.9972861409187317,
  0.997469961643219,
  0.9974114298820496],
 'valid_metrics': [0.9968247413635254,
  0.9971885681152344,
  0.996278703212738,
  0.9974451065063477,
  0.9976120591163635]}

In [84]:
model_1.eval()
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to('cuda'), y.to('cuda')
        y_pred = model_1(X)
        print(y_pred[2])
        print(y[2])
        break

tensor([[ -2.3966,  -5.0995,  -6.8206,  -7.1051,  -7.5369,  -9.8200, -10.7898,
         -11.5938, -11.5923, -12.3831, -12.2055, -12.2148, -12.8696, -12.9159,
         -12.3978, -12.6889, -12.4239, -12.4725, -12.9213, -13.1553, -13.3566,
         -13.8606, -13.9446, -13.8774, -14.1674, -13.8163, -13.8364, -13.8744,
         -14.0042, -14.2044, -14.1366, -14.3223, -14.3573, -14.5248, -14.4208,
         -14.4611, -13.8268, -14.2893, -13.8259, -13.4515, -14.3269, -14.5181,
         -14.2891, -14.0896, -13.8813, -14.0900, -14.1618, -13.8198, -14.0116,
         -14.0565, -13.7579, -13.7061, -14.0336, -13.8919, -13.6371, -13.6872,
         -13.7355, -13.9641, -14.7141, -14.9807, -15.0240, -14.7946, -15.1510,
         -15.1751, -15.4191, -15.3948, -15.3921, -14.3237, -14.7062, -14.6213,
         -15.4907, -14.6492, -13.9406, -14.2876, -15.1717, -15.6093, -15.6981,
         -15.3489, -15.2089, -15.1939, -15.2366, -14.9049, -14.8529, -14.5015,
         -14.2919, -14.1660, -14.4028, -14.3868, -14

In [85]:
metric = torchmetrics.Accuracy(task='binary').to('cuda')
evaluate(model_1, metric, test_loader)

tensor(0.9976, device='cuda:0')